# Saliency incremental threshold calibration

첫 셀 설정 후 **Kernel Restart → Run All** 하면 자료 생성/재사용 → 자료 검증과 가림 실험 진단 → 단일 분할 비교 → 20개 분할 비교를 수행합니다. 현재 모델은 첫 설정 셀 SOURCE_MODEL을 확인하세요. Continuous FIQA 기준입니다.

**Faithfulness 실패는 failed로 표시하지만 실험은 계속합니다.** Saliency 무효 query는 FIQA-only fallback으로 처리합니다. 파일 손상·query 행 누락·모델/분할 불일치 등 자료 검증 실패는 여전히 비교를 차단합니다. 원본 gallery가 필요한 offline 탐색적 실험입니다.

**재실행 안내:** 아래 저장 출력은 변경 전 실행 기록입니다. 입력·결과 파일은 보존했으며, 새 코드 결과는 Kernel Restart → Run All 후 확인하세요. BUILD/RUN/WRITE 설정은 유지하고 SOURCE_MODEL만 바꾸면 모델별로 실행할 수 있습니다.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'research').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs/survface_20260902/20260902-R001-61915edf_step4_survface_arcface-7972a704552df378345f',
    'adaface': PROJECT_ROOT / 'runs/survface_20260830/20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43',
    'magface': PROJECT_ROOT / 'runs/survface_20260831/20260831-R001-6695386d_step4_survface_magface-6931178ad2025e1b3799',
    'edgeface': PROJECT_ROOT / 'runs/survface_20260901/20260901-R001-56c2f3ed_step4_survface_edgeface-a348c305af33c223b337',
}
SOURCE_MODEL = 'edgeface'
FIQA_VARIANT = 'L'
COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
METRIC_CONTRACT = 'genuine-score-topk-v2'
RESULT_ROOT = PROJECT_ROOT / 'results/calibration'
# 01 결과를 참고해 고정한 기준. 같은 test의 02 비교는 탐색적으로 해석합니다.
BASELINE_METHOD = 'continuous_fiqa'  # 또는 continuous_fiqa_margin / continuous_fiqa_margin_distortion
RETRIEVAL_FEATURE_DIR = None       # retrieval 기준 방법이면 01의 완료 feature 경로 필수
SALIENCY_FEATURE_DIR = None        # 자동 생성이면 None 유지; BUILD=False일 때 완료 경로 지정
FAITHFULNESS_DIR = None            # 자동 생성이면 None 유지; BUILD=False일 때 완료 경로 지정

# True: 자료 생성/재개 후 검증·비교까지 진행. 완료 묶음은 재사용합니다.
BUILD_SALIENCY_INPUTS = True
REUSE_TEST_SALIENCY = True
# 중단된 ArcFace 자료의 명시적 복구 경로. 새로운 실험 설정으로 시작할 때만 None으로 변경.
#SALIENCY_RESUME_DIR = (RESULT_ROOT / 'saliency_inputs/20260902-R001-61915edf' /
#    'saliency-inputs-492103c73c999bb998628401') if SOURCE_MODEL == 'arcface' else None

SALIENCY_RESUME_DIR = None
SALIENCY_DEVICE = 'cuda'
GRADCAM_BATCH_SIZE = 4
INPUT_CHUNK_SIZE = 128
FAITHFULNESS_BATCH_SIZE = 32
FAITHFULNESS_MAXIMUM_SAMPLES = 10000  # None이면 가능한 calibration 전체
OCCLUSION_FRACTION = .10
FAITHFULNESS_RANDOM_REPEATS = 5
INPUT_SEED = 8972

PARTITION_SEED = 8972
TARGET_FPIRS = (.01, .05, .10, .20, .30)
SAFETY_FRACTION = .30
KNOT_QUANTILES = (1/3, 2/3)
SMOOTHING = .01
RIDGE = .001
MAX_ITERATIONS = 2000
MARGIN_SLOPE_CAP = .95
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 8972
RUN_INCREMENTAL_CALIBRATION = True
WRITE_INCREMENTAL_RESULTS = True
RUN_SPLIT_STABILITY = True
WRITE_SPLIT_RESULTS = True
SPLIT_SEEDS = (*range(19), 8972)
if SOURCE_MODEL not in SURVFACE_RUN_CANDIDATES or FIQA_VARIANT not in ('S', 'L'):
    raise ValueError('명시된 FR 모델과 FIQA variant를 선택하세요.')
if WRITE_INCREMENTAL_RESULTS and not RUN_INCREMENTAL_CALIBRATION:
    raise ValueError('WRITE requires RUN_INCREMENTAL_CALIBRATION')
if WRITE_SPLIT_RESULTS and not RUN_SPLIT_STABILITY:
    raise ValueError('WRITE requires RUN_SPLIT_STABILITY')
if RUN_SPLIT_STABILITY and (len(SPLIT_SEEDS) < 2 or len(set(SPLIT_SEEDS)) != len(SPLIT_SEEDS)):
    raise ValueError('분할 안정성에는 중복 없는 seed 두 개 이상이 필요합니다.')

if BUILD_SALIENCY_INPUTS and (SALIENCY_FEATURE_DIR is not None or FAITHFULNESS_DIR is not None):
    raise ValueError("자동 생성 또는 기존 두 입력 경로 지정 중 하나를 선택하세요.")
if not BUILD_SALIENCY_INPUTS and ((SALIENCY_FEATURE_DIR is None) != (FAITHFULNESS_DIR is None)):
    raise ValueError("기존 saliency/faithfulness 경로를 함께 지정하세요.")


## 1. 방법과 실행 조건

- 기준은 Continuous FIQA 계열이며 같은 설정으로 calibration에서 재적합합니다. 01의 test 결과를 참고한 기준 선택이므로 현재 결과는 탐색적입니다.
- 비교 방법: 기준, +outside_face_attention(얼굴 밖 집중 비율), +saliency_entropy(집중 분포 엔트로피), +두 특징. 스케일·knot·회귀는 calibration fit non-mated, 안전 보정은 held-out safety에서만 적합합니다.
- High−Low와 High−Random의 identity-cluster 95% CI 하한이 모두 양수여야 strong faithfulness입니다. **이 기준은 그대로 평가하되 통과 여부로 성능 비교를 차단하지 않습니다.** 실패도 결과 표와 manifest에 저장됩니다.
- 가림 실험 score drop/Random은 threshold 입력이 아닙니다. 성능 개선이 나타나더라도 faithfulness 또는 압축오차의 인과관계를 입증한 것은 아닙니다.
- 자료 검증은 계속 필수입니다: calibration/test 모든 query 행과 alignment hash, split별 label-free origin-top1 gallery, 모델·입력 lineage, 파일 SHA, 가림 실험의 test 비중복을 확인합니다. 행 누락·손상은 차단합니다.
- 기록된 heatmap unavailable/invalid는 FIQA-only fallback으로 처리합니다. 세 saliency 방법 모두 두 특징의 공통 유효 mask를 사용합니다. NaN을 0으로 대치하거나 query를 삭제하지 않습니다.
- 기존 identity 기반 fit/safety 분할을 그대로 유지합니다. Saliency는 유효 fit non-mated로만 적합하고, 두 경로를 합친 규칙을 safety non-mated 전체에서 보정합니다. 유효 fit 표본이 20개 미만이면 그 방법은 전체 FIQA-only로 처리하고 이유를 기록합니다.
- 선택한 기준이 +margin/+distortion이어도 saliency 무효 query의 fallback은 순수 Continuous FIQA입니다. 전체 test의 FPIR/TPIR와 fallback 경로별 분모·실패 수를 함께 보고합니다.
- 원본 gallery 접근이 필요하므로 압축 DB만으로 배포 가능한 방법이라고 주장하지 않습니다.


In [2]:
import json
import pandas as pd
from IPython.display import display, Markdown
from research.runtime.hashing import sha256_file
from research.experiments.fiqa_threshold_calibration import load_condition_score_artifact
from research.experiments.fiqa_retrieval_features import load_retrieval_features
from research.experiments.saliency_incremental_calibration import (
    load_saliency_incremental_inputs, assess_incremental_gate,
    run_saliency_incremental_calibration, write_saliency_incremental_result,
)
from research.fiqa import CRFIQA_VARIANTS, load_fiqa_score_artifact

source_dir = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
source = json.loads((source_dir / 'run_manifest.json').read_text(encoding='utf8'))
if source.get('status') != 'completed' or not (source_dir / 'COMPLETED').is_file():
    raise ValueError('완료 source run이 필요합니다.')
run_id = source['run_id']
condition_dir = RESULT_ROOT / 'condition_scores' / run_id / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}' / METRIC_CONTRACT
condition = load_condition_score_artifact(condition_dir)
for key, expected in {
    'source_run_id': run_id, 'model_uid': source['config']['model_uid'],
    'compression_profile': COMPRESSION_PROFILE, 'search_mode': SEARCH_MODE,
    'metric_contract': METRIC_CONTRACT, 'source_run_manifest_sha256': sha256_file(source_dir / 'run_manifest.json'),
}.items():
    if condition.manifest.get(key) != expected:
        raise ValueError(f'condition lineage 불일치: {key}')


## 1.1 자료 생성과 가림 실험
Calibration 특징을 생성하고 기존 test 특징은 검증 후 재사용합니다. 기본 가림 실험 표본은 calibration 10,000장입니다. 진행률과 완료 경로가 아래에 표시됩니다.


In [3]:
from research.experiments.saliency_calibration_inputs import build_saliency_calibration_inputs
input_bundle = None
if BUILD_SALIENCY_INPUTS:
    input_bundle = build_saliency_calibration_inputs(
        source_dir, condition, RESULT_ROOT / 'saliency_inputs' / run_id,
        device=SALIENCY_DEVICE, reuse_test_saliency=REUSE_TEST_SALIENCY,
        gradcam_batch_size=GRADCAM_BATCH_SIZE, chunk_size=INPUT_CHUNK_SIZE,
        faithfulness_batch_size=FAITHFULNESS_BATCH_SIZE,
        faithfulness_maximum_samples=FAITHFULNESS_MAXIMUM_SAMPLES,
        occlusion_fraction=OCCLUSION_FRACTION, random_repeats=FAITHFULNESS_RANDOM_REPEATS,
        seed=INPUT_SEED, bootstrap_repeats=BOOTSTRAP_RESAMPLES,
        progress=lambda event: print(event, flush=True), resume_from=SALIENCY_RESUME_DIR,
    )
    saliency_dir = input_bundle['saliency_directory']
    faithfulness_dir = input_bundle['faithfulness_directory']
else:
    saliency_dir = Path(SALIENCY_FEATURE_DIR) if SALIENCY_FEATURE_DIR is not None else source_dir / 'artifacts/step2_workflow/saliency_population'
    faithfulness_dir = Path(FAITHFULNESS_DIR) if FAITHFULNESS_DIR is not None else PROJECT_ROOT / 'results/paper/survface' / run_id / 'faithfulness_v2_all'
inputs = load_saliency_incremental_inputs(condition, saliency_dir, faithfulness_dir)
gate = assess_incremental_gate(condition, inputs)
display(pd.DataFrame([{'saliency_directory': str(saliency_dir),
                      'faithfulness_directory': str(faithfulness_dir), 'baseline': BASELINE_METHOD}]))


{'stage': 'preflight', 'torch': '2.7.1+cu118', 'cuda_runtime': '11.8', 'device': 'cuda', 'device_name': 'NVIDIA GeForce GTX 1080 Ti'}
{'stage': 'source_verified', 'calibration_queries': 160408, 'test_queries': 182159, 'faithfulness_samples': 10000, 'reuse_test_saliency': True}
{'stage': 'reuse_completed', 'directory': 'C:\\ronbun\\results\\calibration\\saliency_inputs\\20260901-R001-56c2f3ed\\saliency-inputs-45dffd1416a3a8c075660f2e'}


,saliency_directory,faithfulness_directory,baseline
0,C:\ronbun\results\calibration\saliency_inputs\...,C:\ronbun\results\calibration\saliency_inputs\...,continuous_fiqa


## 2. 자료 검증과 faithfulness를 별도로 표시

자료 검증이 passed이면 성능 비교가 가능합니다. Faithfulness는 passed/failed와 그 이유를 따로 표시합니다. High/Low/Random은 중요한/덜 중요한/무작위 영역을 가렸을 때 원본 cosine score가 감소한 양입니다.


In [4]:
display(pd.DataFrame([{
    'data_validation': gate['data_validation_status'],
    'faithfulness': gate['faithfulness_status'],
    'faithfulness_policy': gate['faithfulness_policy'],
    'comparison_enabled': gate['comparison_enabled'],
    'calibration_coverage': gate['calibration_coverage'],
    'test_coverage': gate['test_coverage'],
    'calibration_fallback_count': gate['invalid_saliency_counts'].get('calibration'),
    'test_fallback_count': gate['invalid_saliency_counts'].get('test'),
    'fallback_policy': gate['fallback_policy'],
    'faithfulness_test_overlap': gate['faithfulness_test_overlap'],
}]))
display(inputs['faithfulness_summary'].loc[inputs['faithfulness_summary'].group.eq('all'),
    ['metric', 'sample_count', 'mean', 'mean_ci_lower', 'mean_ci_upper']])
display(pd.DataFrame([gate['faithfulness']]))
if gate['warnings']:
    display(pd.DataFrame({'diagnostic_warning': gate['warnings']}))
if gate['reasons']:
    display(pd.DataFrame({'data_blocked_reason': gate['reasons']}))
    display(Markdown('**자료 검증 실패:** 성능 비교를 차단합니다.'))
elif gate['faithfulness_status'] == 'failed':
    display(Markdown('**Faithfulness: failed.** 실패를 그대로 기록하고 네 방법의 성능 비교를 계속합니다.'))
else:
    display(Markdown('자료 검증 및 faithfulness 통과. RUN 설정에 따라 성능 비교를 진행합니다.'))


,data_validation,faithfulness,faithfulness_policy,comparison_enabled,calibration_coverage,test_coverage,calibration_fallback_count,test_fallback_count,fallback_policy,faithfulness_test_overlap
0,passed,failed,diagnostic_only,True,0.999202,0.999182,128,149,invalid-saliency-fiqa-only-joint-safety-v1,0


,metric,sample_count,mean,mean_ci_lower,mean_ci_upper
0,high_saliency_occlusion_score_drop,10000,0.083153,0.081121,0.085170
1,low_saliency_occlusion_score_drop,10000,0.140980,0.137865,0.144447
2,random_occlusion_score_drop,10000,0.091776,0.088832,0.094736
3,faithfulness_gain_over_low_saliency,10000,-0.057827,-0.061320,-0.054420
4,faithfulness_gain_over_random,10000,-0.008623,-0.011774,-0.005551


,status,strong_faithfulness_pass,high_over_low_mean,high_over_low_ci_lower,high_over_low_ci_upper,high_over_random_mean,high_over_random_ci_lower,high_over_random_ci_upper,gated_high_low_contrast,group,random_control_role,random_is_threshold_feature,reasons,role
0,failed,False,-0.057827,-0.06132,-0.05442,-0.008623,-0.011774,-0.005551,0.0,all,faithfulness_negative_control_only,False,[High does not exceed Low with a strictly posi...,diagnostic_only


,diagnostic_warning
0,High does not exceed Low with a strictly posit...
1,High does not exceed Random with a strictly po...
2,calibration: 128 invalid saliency queries use ...
3,test: 149 invalid saliency queries use FIQA-on...


**Faithfulness: failed.** 실패를 그대로 기록하고 네 방법의 성능 비교를 계속합니다.

## 3. FIQA 대비 saliency 추가 비교

자료 검증을 통과하면 faithfulness 성공/실패와 무관하게 네 방법을 비교·저장합니다. Faithfulness 실패는 성능 표와 별도 진단 CSV에 남습니다. 표의 faithfulness_status는 공통 saliency 진단 상태이며, 기준 FIQA 자체의 성능 판정이 아닙니다.

FPIR은 query 단위, TPIR20은 정답 score threshold 통과 AND rank≤20이며 mated identity cluster CI를 사용합니다. 실제 FPIR과 목표 충족 여부를 같이 읽으세요. TPIR 상승만으로 개선을 단정하지 않습니다. 비율 차이 ×100은 %p입니다.


In [5]:
comparison = stability = None
comparison_path = stability_path = None
fiqa = retrieval = None
if (RUN_INCREMENTAL_CALIBRATION or RUN_SPLIT_STABILITY) and gate['comparison_enabled']:
    fiqa = load_fiqa_score_artifact(RESULT_ROOT / 'fiqa_scores/survface' / CRFIQA_VARIANTS[FIQA_VARIANT].model_uid)
    if BASELINE_METHOD != 'continuous_fiqa':
        if RETRIEVAL_FEATURE_DIR is None:
            raise ValueError('01의 완료 retrieval feature 경로가 필요합니다.')
        retrieval = load_retrieval_features(RETRIEVAL_FEATURE_DIR, condition)
    settings = dict(
        baseline_method=BASELINE_METHOD, retrieval=retrieval, target_fpirs=TARGET_FPIRS,
        safety_fraction=SAFETY_FRACTION, knot_quantiles=KNOT_QUANTILES, smoothing=SMOOTHING,
        ridge=RIDGE, max_iterations=MAX_ITERATIONS, margin_slope_cap=MARGIN_SLOPE_CAP,
        resamples=BOOTSTRAP_RESAMPLES, bootstrap_seed=BOOTSTRAP_SEED,
    )
    if RUN_INCREMENTAL_CALIBRATION:
        comparison = run_saliency_incremental_calibration(condition, fiqa, inputs, partition_seeds=(PARTITION_SEED,), **settings)
        if WRITE_INCREMENTAL_RESULTS:
            comparison_path = write_saliency_incremental_result(RESULT_ROOT / 'saliency_incremental' / run_id, comparison)
        display(comparison['method_summary'][[
            'target_fpir', 'method', 'realized_fpir', 'fpir_wilson95_low', 'fpir_wilson95_high',
            'tpir_at_rank_k', 'tpir_cluster95_low', 'tpir_cluster95_high', 'target_met_on_test', 'fallback_query_count', 'fallback_query_fraction', 'faithfulness_status']])
elif not gate['comparison_enabled']:
    display(Markdown('**자료 검증 실패로 보정 차단:** baseline/+Saliency 적합과 성능 결과 저장을 수행하지 않았습니다.'))
else:
    display(Markdown('RUN_INCREMENTAL_CALIBRATION=False: readiness 진단만 수행했습니다.'))


,target_fpir,method,realized_fpir,fpir_wilson95_low,fpir_wilson95_high,tpir_at_rank_k,tpir_cluster95_low,tpir_cluster95_high,target_met_on_test,fallback_query_count,fallback_query_fraction,faithfulness_status
0,0.01,baseline,0.010630,0.010069,0.011221,0.000794,0.000438,0.001280,False,0,0.000000,failed
1,0.01,plus_outside,0.010999,0.010429,0.011601,0.000662,0.000339,0.001104,False,149,0.000818,failed
2,0.01,plus_entropy,0.010556,0.009997,0.011145,0.000844,0.000476,0.001336,False,149,0.000818,failed
3,0.01,plus_both,0.011073,0.010501,0.011677,0.000728,0.000390,0.001176,False,149,0.000818,failed
4,0.05,baseline,0.051530,0.050302,0.052786,0.003608,0.002664,0.004712,False,0,0.000000,failed
5,0.05,plus_outside,0.051094,0.049871,0.052345,0.003658,0.002691,0.004757,False,149,0.000818,failed
6,0.05,plus_entropy,0.051686,0.050456,0.052943,0.003873,0.002894,0.005027,False,149,0.000818,failed
7,0.05,plus_both,0.050905,0.049685,0.052154,0.003740,0.002761,0.004888,False,149,0.000818,failed
8,0.10,baseline,0.097523,0.095869,0.099202,0.008656,0.006644,0.011002,True,0,0.000000,failed
9,0.10,plus_outside,0.098681,0.097018,0.100369,0.008904,0.006830,0.011229,True,149,0.000818,failed


In [6]:
if comparison is not None:
    display(comparison['paired_comparisons'])
    display(Markdown(f'결과 경로: {comparison_path or "메모리 내 계산만 수행"}'))


,reference_successes,candidate_successes,both_successes,total,candidate_minus_reference,paired_bootstrap95_low,paired_bootstrap95_high,partition_seed,target_fpir,reference_method,...,resamples,bootstrap_seed,reference_realized_fpir,candidate_realized_fpir,candidate_fallback_query_count,candidate_target_met,faithfulness_status,strong_faithfulness_pass,faithfulness_policy,fallback_policy
0,1294,1339,1092,121736,0.000370,0.000016,0.000698,8972,0.01,baseline,...,2000,8972,0.010630,0.010999,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
1,48,40,37,60423,-0.000132,-0.000263,-0.000017,8972,0.01,baseline,...,2000,8972,0.010630,0.010999,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
2,1294,1285,1160,121736,-0.000074,-0.000337,0.000181,8972,0.01,baseline,...,2000,8972,0.010630,0.010556,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
3,48,51,46,60423,0.000050,-0.000033,0.000136,8972,0.01,baseline,...,2000,8972,0.010630,0.010556,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
4,1294,1348,1021,121736,0.000444,0.000033,0.000830,8972,0.01,baseline,...,2000,8972,0.010630,0.011073,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
5,48,44,38,60423,-0.000066,-0.000203,0.000055,8972,0.01,baseline,...,2000,8972,0.010630,0.011073,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
6,6273,6220,5536,121736,-0.000435,-0.001060,0.000197,8972,0.05,baseline,...,2000,8972,0.051530,0.051094,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
7,218,221,182,60423,0.000050,-0.000279,0.000382,8972,0.05,baseline,...,2000,8972,0.051530,0.051094,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
8,6273,6292,5821,121736,0.000156,-0.000361,0.000641,8972,0.05,baseline,...,2000,8972,0.051530,0.051686,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
9,218,234,210,60423,0.000265,0.000067,0.000479,8972,0.05,baseline,...,2000,8972,0.051530,0.051686,149,False,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1


결과 경로: C:\ronbun\results\calibration\saliency_incremental\20260901-R001-56c2f3ed\saliency-incremental-74d07d8c4c22ea29580c1de8

## 4. 선택적 분할 안정성

RUN_SPLIT_STABILITY=True이면 동일 test/gallery에서 calibration fit/safety만 다시 나눕니다. 미리 정한 seed 전체를 사용하며 최적 seed를 선택하지 않습니다. 최소·중앙·최대는 기술통계이며 독립 실험 반복이나 CI가 아닙니다. 자료 검증이 실패한 경우에만 이 단계도 차단됩니다. Faithfulness 실패만으로는 차단하지 않습니다.


In [7]:
if RUN_SPLIT_STABILITY and gate['comparison_enabled']:
    stability = run_saliency_incremental_calibration(condition, fiqa, inputs, partition_seeds=SPLIT_SEEDS, **settings)
    if WRITE_SPLIT_RESULTS:
        stability_path = write_saliency_incremental_result(RESULT_ROOT / 'saliency_incremental_split_stability' / run_id, stability)
    display(stability['split_summary'])


,method,target_fpir,split_count,target_met_split_count,fpir_min,fpir_median,fpir_max,tpir_min,tpir_median,tpir_max,fallback_count_min,fallback_count_max,fallback_fraction_max,faithfulness_status,strong_faithfulness_pass,faithfulness_policy,fallback_policy
0,baseline,0.01,20,1,0.009915,0.010654,0.011098,0.000728,0.000778,0.000877,0,0,0.000000,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
1,baseline,0.05,20,3,0.048400,0.050725,0.052055,0.003409,0.003591,0.003757,0,0,0.000000,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
2,baseline,0.10,20,20,0.090877,0.098102,0.099371,0.007994,0.008838,0.008970,0,0,0.000000,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
3,baseline,0.20,20,20,0.185311,0.194511,0.198076,0.020721,0.021565,0.022061,0,0,0.000000,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
4,baseline,0.30,20,20,0.279416,0.291270,0.295336,0.033050,0.034532,0.035003,0,0,0.000000,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
5,plus_both,0.01,20,2,0.009849,0.010888,0.011131,0.000662,0.000720,0.000778,149,149,0.000818,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
6,plus_both,0.05,20,6,0.047866,0.050581,0.051850,0.003591,0.003782,0.003922,149,149,0.000818,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
7,plus_both,0.10,20,18,0.090343,0.098677,0.100324,0.008391,0.009301,0.009599,149,149,0.000818,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
8,plus_both,0.20,20,20,0.185105,0.195201,0.198963,0.020108,0.021258,0.021697,149,149,0.000818,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1
9,plus_both,0.30,20,20,0.280180,0.291504,0.295853,0.033001,0.034366,0.034871,149,149,0.000818,failed,False,diagnostic_only,invalid-saliency-fiqa-only-joint-safety-v1


## 5. 결과와 재실행

입력은 results/calibration/saliency_inputs/<run_id>/<UID>에 저장됩니다. BUILD_SALIENCY_INPUTS=True를 유지하면 완료 묶음을 검증해 재사용하고, 중단된 생성은 완료 chunk 다음부터 재개합니다.

BUILD_SALIENCY_INPUTS=False로 재사용하려면 생성 셀에 표시된 두 완료 디렉터리를 첫 셀에 지정하세요. 둘 다 None이면 과거 자료 진단만 수행합니다.

기본 가림 실험은 사전 고정 calibration 10,000장입니다. Test와 ID/identity/이미지 내용이 중복되는 표본을 제외하고 선정 기록을 저장합니다. Identity-cluster CI를 원자료에서 다시 계산합니다. High−Low/High−Random 기준 실패는 failed로 기록하며 성능 비교는 계속합니다. 결과를 보고 표본·가림 비율·seed를 바꿔 통과시키지 않습니다.

성능 결과는 saliency_incremental 및 saliency_incremental_split_stability에 새 UID로 저장합니다. 현재 01의 test 결과를 참고해 기준을 정했으므로 02도 탐색적 결과입니다. 다중 비교·calibration uncertainty·미사용 test의 확증은 남아 있습니다.

Windows 저장 오류 복구: SALIENCY_RESUME_DIR의 기존 자료를 검증해 새 구현 UID로 복사합니다. 원본은 변경하지 않습니다. 생성 설정을 의도적으로 바꾸는 새 실험에서는 이 변수를 None으로 지정하세요.

각 결과 디렉터리의 faithfulness_diagnostics.csv와 manifest에는 실패 이유와 원래 효과/CI가 남습니다. method_summary.csv, paired_comparisons.csv, split_summary.csv에도 faithfulness_status를 기록합니다. 성능 개선과 faithfulness는 별개의 결론입니다.

새 결과는 schema_version=3입니다(metric_contract는 genuine-score-topk-v2 유지). fallback_diagnostics.csv에는 fit/safety/test 경로별 query 수·FPIR 및 test TPIR가 저장됩니다. Calibration에는 정답 점수가 없어 TPIR는 빈 값(tpir_available=False)입니다. fallback_queries.csv에는 유효하지 않은 saliency query 목록이 저장됩니다. 전체 집계는 method_summary.csv 및 split_summary.csv의 fallback 필드로 확인하세요. 경로별 비율은 소표본 기술통계이며 전체 paired CI를 대체하지 않습니다.


In [8]:
input_path = str(input_bundle['directory']) if input_bundle is not None else str(saliency_dir)
display(pd.DataFrame([
    {'stage': 'data validation', 'status': gate['data_validation_status'], 'artifact': input_path},
    {'stage': 'faithfulness (diagnostic only)', 'status': gate['faithfulness_status'], 'artifact': str(faithfulness_dir)},
    {'stage': 'single split', 'status': 'computed' if comparison is not None else ('blocked' if not gate['comparison_enabled'] else 'not_run'), 'artifact': str(comparison_path) if comparison_path else None},
    {'stage': 'split stability', 'status': 'computed' if stability is not None else ('blocked' if not gate['comparison_enabled'] else 'not_run'), 'artifact': str(stability_path) if stability_path else None},
]))


,stage,status,artifact
0,data validation,passed,C:\ronbun\results\calibration\saliency_inputs\...
1,faithfulness (diagnostic only),failed,C:\ronbun\results\calibration\saliency_inputs\...
2,single split,computed,C:\ronbun\results\calibration\saliency_increme...
3,split stability,computed,C:\ronbun\results\calibration\saliency_increme...
